# Bài 8 · Xử lý dữ liệu thời gian

**Lập trình xử lý dữ liệu · 2627-1 · Viện TTNT, UET-VNU**

Chọn **Save a copy in Drive** trước khi sửa.

## Mục tiêu bài học

1. Đọc đúng ngày giờ và tính khoảng thời gian giữa hai ngày.
2. Tổng hợp theo lịch và giải thích cách tính cửa sổ trượt.
3. Chọn kỳ gốc phù hợp; nhận diện kỳ chưa trọn.
4. Kiểm chứng kết quả bằng bảng nhỏ và dữ liệu gốc.

**Giáo trình:** [McKinney, chương 11 — Time Series](https://wesmckinney.com/book/time-series).
Các bảng nhỏ là **giả lập**. Dữ liệu thật là đánh giá tại Santiago,
snapshot **29/06/2026**. Mỗi hàng trong `reviews.csv` là một đánh giá;
`date` là ngày ghi nhận đánh giá, `listing_id` là mã chỗ ở.
Code và output được kiểm bằng pandas **3.0.3**.

## Chuẩn bị

Ô cài đặt dành cho Colab. Nếu phiên Colab đã nạp pandas phiên bản khác, chọn
**Restart session and run all** sau khi cài. Chạy lần lượt các ô demo; những ô có nhãn
**Bài tập** để bạn tự viết lời giải. Dữ liệu được tải ở ô dưới nhưng
chưa lọc ngày hoặc đổi kiểu; các bước này sẽ được thực hiện trong bài.

In [ ]:
%pip -q install pandas==3.0.3

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

URL_REVIEWS = ("https://data.insideairbnb.com/chile/rm/santiago/"
               "2026-06-29/visualisations/reviews.csv")
tep = Path("data/reviews_santiago_full_2026-06-29.csv.gz")
nguon = tep if tep.exists() else URL_REVIEWS
rv_raw = pd.read_csv(nguon, usecols=["listing_id", "date"])
print("pandas:", pd.__version__)
print("Số đánh giá trong file gốc:", len(rv_raw))

## 1. Đọc và tính toán với ngày giờ

Một chuỗi giữ cách viết ngày, còn kiểu ngày giờ cho phép sắp xếp theo thời gian, lấy thành phần của ngày và trừ hai ngày. Phần này bám McKinney §11.1; múi giờ tham khảo §11.4.

### Đổi chuỗi thành ngày giờ

Không trừ trực tiếp hai chuỗi được: Python báo `TypeError`, vì chuỗi chỉ là văn bản. Sắp xếp theo chuỗi còn đặt `03/03/2026` trước `28/02/2026`. `to_datetime` đổi chuỗi thành ngày giờ. `format="%d/%m/%Y"` mô tả thứ tự ngày/tháng/năm; `%d` là ngày, `%m` là tháng, `%Y` là năm bốn chữ số. Một giá trị ngày giờ riêng lẻ có kiểu `Timestamp`.

In [ ]:
chuoi = pd.Series(["28/02/2026", "03/03/2026"])
ngay = pd.to_datetime(chuoi, format="%d/%m/%Y")
print(ngay)

### Ngày và tháng có thể bị đảo

`03/07/2026` có thể là 3/7 hoặc 7/3. Không thể xác định chỉ bằng cách xem chuỗi này. Tra quy ước của nguồn; khi ghi dữ liệu mới có thể dùng `YYYY-MM-DD` để tránh mơ hồ.

In [ ]:
print(pd.Series({
    "ngày/tháng/năm": pd.to_datetime(
        "03/07/2026", format="%d/%m/%Y"),
    "tháng/ngày/năm": pd.to_datetime(
        "03/07/2026", format="%m/%d/%Y"),
}))

### Ngày không hợp lệ và ngày bị thiếu

Mặc định `errors="raise"` báo lỗi nếu không đọc được ngày. `errors="coerce"` cho phép tiếp tục bằng cách chuyển giá trị đó thành `NaT`. Ngày bị thiếu từ nguồn cũng thành `NaT`, nên giữ cột gốc để phân biệt hai trường hợp.

In [ ]:
goc = pd.Series(["29/06/2026", "31/06/2026", None])
doc = pd.to_datetime(
    goc, format="%d/%m/%Y", errors="coerce")
print(pd.DataFrame({"goc": goc, "ngay": doc}))

Phân biệt ngày không đọc được với ngày đã thiếu từ nguồn:

In [ ]:
loi_doc = goc.notna() & doc.isna()
print(goc[loi_doc])

### Lấy tháng và thứ trong tuần

Giống `.str` ở Bài 7, dùng `.dt` trên một Series ngày giờ. `.dt.month` lấy tháng; `.dt.dayofweek` lấy thứ, từ 0 (thứ Hai) đến 6 (Chủ nhật). `.dt.year` và `.dt.quarter` lấy năm và quý.

In [ ]:
print(pd.DataFrame({
    "ngay": ngay,
    "thang": ngay.dt.month,
    "thu": ngay.dt.dayofweek,
}))

### Tính khoảng cách giữa hai ngày

Mỗi giá trị ngày giờ là một `Timestamp`. Hiệu của hai `Timestamp` là một `Timedelta` (khoảng thời gian); ô dưới in `3 days 00:00:00`. `.days` lấy phần ngày. Với một Series chứa các khoảng thời gian, dùng `.dt.days`. Ở đây dữ liệu chỉ có ngày; nếu có giờ và cần ngày dạng số thực, dùng `.dt.total_seconds() / 86400`.

In [ ]:
khoang = ngay.iloc[1] - ngay.iloc[0]
print(khoang)

In [ ]:
print(khoang.days)

### Bài tập 1: Tính số đêm lưu trú

Mỗi hàng là một lượt đặt. Ngày ghi theo **DD/MM/YYYY**. Một bạn viết code ở ô dưới;
chạy ô đó và xem kết quả có hợp lý với các ngày đã cho không.

**Sửa code để tính đúng số đêm của từng lượt đặt.**

In [ ]:
dat = pd.DataFrame({
    "vao": ["03/07/2026", "10/07/2026"],
    "ra": ["06/07/2026", "12/07/2026"],
})
vao = pd.to_datetime(dat["vao"])
ra = pd.to_datetime(dat["ra"])
so_dem = (ra - vao).dt.days
print(so_dem.tolist())

In [ ]:
# TODO: viết lời giải ở đây.

### Đổi múi giờ 📖 Tự học

Nhật ký (log) của phần mềm hoặc API có thể ghi thời điểm theo UTC (giờ phối hợp quốc tế), giờ chuẩn dùng chung cho mọi múi giờ. `tz="UTC"` khai báo múi giờ cho thời điểm đã biết là UTC; `tz_convert` đổi sang múi giờ Việt Nam, không đổi thời điểm thực. Ngày đánh giá trong Inside Airbnb không có giờ; không tự gán UTC cho cột này.

In [ ]:
utc = pd.Timestamp("2026-06-29 22:30", tz="UTC")
print(utc.tz_convert("Asia/Ho_Chi_Minh"))

## 2. Tổng hợp theo thời gian

Muốn xem số đánh giá thay đổi theo thời gian, cần gộp các hàng vào những khoảng trên lịch. Phần này bám McKinney §11.2, §11.6 và §11.7.

### Dữ liệu đánh giá tại Santiago

`rv_raw` là bảng gốc đã tải trong phần chuẩn bị. `date` dùng định dạng năm-tháng-ngày; đổi kiểu trước khi lọc và đặt chỉ mục. Đây là ngày ghi nhận đánh giá, không phải ngày nhận phòng. `read_csv(parse_dates=["date"])` ở Bài 6 đổi kiểu ngay khi đọc; ở đây tách riêng để thấy `format`.

In [ ]:
rv_raw["date"] = pd.to_datetime(
    rv_raw["date"], format="%Y-%m-%d")
print(rv_raw.head(3))

### Kiểm tra ngày vượt mốc chụp

`pd.Timestamp` tạo mốc chụp 29/06/2026 để so với cột `date`. Kiểm trước khi lọc: 155 hàng ghi ngày 30/6 và 49 hàng ghi ngày 1/7, tổng 204. Bản chụp được thu trong bốn ngày, từ 29/06 đến 02/07 (cột `last_scraped` của bảng chỗ ở), nên chỗ ở thu muộn có đánh giá sau 29/06. Bài này giới hạn đến 29/6 nên loại chúng; vẫn giữ `rv_raw` để đối chiếu.

In [ ]:
moc = pd.Timestamp("2026-06-29")
ngoai_moc = rv_raw.loc[rv_raw["date"] > moc, "date"]
print(ngoai_moc.value_counts().sort_index())

### Chọn dữ liệu theo năm và tháng

Dòng đầu giữ các hàng đến mốc chụp, tức loại 204 hàng ở bước trước. `set_index("date")` đặt ngày làm chỉ mục; `sort_index()` sắp theo thời gian. Các đánh giá trong file gốc xếp chủ yếu theo từng chỗ ở, không theo ngày, nên thiếu `sort_index()` thì cắt theo khoảng như `.loc["2026-01":"2026-05"]` báo `KeyError`. Với `DatetimeIndex`, `.loc["2026"]` chọn cả năm và `.loc["2026-03"]` chọn cả tháng. Chỉ mục có thể có nhiều hàng cùng một ngày; không gộp hoặc xoá chúng.

In [ ]:
rv = rv_raw.loc[rv_raw["date"] <= moc].copy()
r = rv.set_index("date").sort_index()
print(pd.Series({
    "năm 2026": len(r.loc["2026"]),
    "tháng 3/2026": len(r.loc["2026-03"]),
}))

### Gộp đánh giá theo tháng

`resample` gộp các hàng theo khoảng thời gian trên lịch. `ME` là tháng, nhãn cuối tháng; `.size()` đếm số hàng. Kết quả giữ cả tháng 2 nằm giữa hai tháng có dữ liệu, với số đếm bằng 0. Với dữ liệu đã là số tiền hoặc số đếm, chọn `.sum()` hoặc `.mean()` theo yêu cầu, không đếm hàng bằng `.size()`. Sách McKinney và nhiều ví dụ cũ viết `"M"`, `"Q"`; pandas 3 báo `ValueError` và yêu cầu `"ME"`, `"QE"`.

In [ ]:
vi_du = pd.Series([1, 1, 1], index=pd.to_datetime([
    "2026-01-03", "2026-01-10", "2026-03-02",
]))
print(vi_du.resample("ME").size())

### Đếm số đánh giá từng tháng

Đếm số hàng, vì mỗi hàng là một đánh giá. Các mã tần suất khác: `D` theo ngày; `W-SUN` theo tuần kết thúc Chủ nhật; `QE` theo quý, nhãn cuối quý. Nhãn cuối kỳ không có nghĩa kỳ đó đã trọn.

In [ ]:
theo_thang = r.resample("ME").size()
print(theo_thang.tail(3))

### Tháng cuối chưa trọn

Tháng 6/2026 chưa trọn ở mốc chụp 29/6. `moc.replace(day=1)` lấy 1/6; chỉ giữ các tháng có nhãn cuối tháng trước 1/6. Tháng 5/2026 là tháng trọn cuối cùng. Chọn kỳ theo mốc chụp, không xoá hàng cuối một cách máy móc bằng `iloc[:-1]`.

In [ ]:
dau_thang = moc.replace(day=1)
thang_day_du = theo_thang.loc[
    theo_thang.index < dau_thang]
print(thang_day_du.tail(2))

### Bài tập 2: Trung bình mỗi ngày

Hệ thống đã thu đủ dữ liệu từ **01–07/06/2026**. `dem_ngay` chỉ liệt kê ngày có đánh giá (giả lập).

**Tính trung bình số đánh giá mỗi ngày trong tuần này.**

In [ ]:
dem_ngay = pd.Series([2, 1, 4], index=pd.to_datetime([
    "2026-06-01", "2026-06-03", "2026-06-07",
]))

In [ ]:
# TODO: viết lời giải ở đây.

### Tính trung bình theo cửa sổ trượt

Để bớt dao động, mỗi tháng lấy trung bình của tháng đó và hai tháng trước. Cửa sổ trượt (`rolling`) dời xuống một hàng mỗi lần: tại T3 gồm T1–T3, tại T4 gồm T2–T4. Tính tay: T3 = (10 + 50 + 30) / 3 = 30; T4 = (50 + 30 + 40) / 3 = 40. `min_periods=3` yêu cầu đủ 3 giá trị không thiếu, nên hai hàng đầu có `NaN`. Kết quả gắn ở cuối cửa sổ (`center=False` mặc định). Chỉ khi mỗi hàng là một tháng thì cửa sổ 3 hàng mới ứng với 3 tháng.

In [ ]:
s = pd.Series([10, 50, 30, 40], index=["T1", "T2", "T3", "T4"])
tb = s.rolling(3, min_periods=3).mean()
print(pd.DataFrame({"so_danh_gia": s, "tb_3_thang": tb}))

### Trung bình của sáu tháng gần nhất

Một hàng của `thang_day_du` là một tháng, nên cửa sổ 6 hàng là 6 tháng. Biểu đồ dùng giai đoạn 2016 đến tháng 5/2026. Đường làm mượt bớt dao động nhưng phản ứng chậm hơn khi số đánh giá đổi nhanh. Ô vẽ bên dưới chỉ để quan sát; cách vẽ học ở Bài 12.

In [ ]:
muot = thang_day_du.rolling(6, min_periods=6).mean()

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
thang_day_du.loc["2016":].plot(ax=ax, color="0.65",
    label="Số đánh giá mỗi tháng")
muot.loc["2016":].plot(ax=ax, label="Trung bình 6 tháng gần nhất")
ax.set_xlabel("")
ax.set_ylabel("Số đánh giá")
ax.legend()
plt.show()

## 3. So sánh giữa các kỳ

Phần này dùng phép dịch hàng (`shift`, McKinney §11.3) để đặt kỳ gốc cạnh kỳ hiện tại. Cần xác định đang so với tháng trước, cùng tháng năm trước hay tổng của một giai đoạn.

### Đặt giá trị kỳ trước cạnh kỳ này

`shift(1)` dời giá trị xuống một hàng: hàng hiện tại nhận giá trị của hàng ngay trên, nhãn giữ nguyên. Sau đó phép trừ hoặc chia ghép hai giá trị theo cùng nhãn như Bài 5.

In [ ]:
mau = pd.Series([10, 12, 15], index=["T1", "T2", "T3"])
print(pd.DataFrame({
    "ky_nay": mau, "ky_truoc": mau.shift(1),
}))

### Thiếu một tháng thì shift bị lệch

`shift(1)` chỉ là tháng trước nếu các hàng theo đúng lịch tháng liên tiếp. Một tháng không thu thập dữ liệu là giá trị chưa biết, khác với 0 đánh giá. Có thể bổ sung nhãn tháng bằng `asfreq("ME")`, giữ `NaN` cho tháng chưa biết.

In [ ]:
thieu = pd.Series([10, 15, 18], index=pd.to_datetime([
    "2026-01-31", "2026-03-31", "2026-04-30",
]))
print(pd.DataFrame({"gia_tri": thieu, "shift_1": thieu.shift(1)}))

Giữ lại tháng chưa có dữ liệu bằng `asfreq("ME")` (đưa về lịch tháng, không tổng hợp). Khi đó tháng 3 nhận `NaN` vì tháng 2 chưa biết:

In [ ]:
lich = thieu.asfreq("ME")
print(pd.DataFrame({"gia_tri": lich, "shift_1": lich.shift(1)}))

### Tính tỷ lệ thay đổi

`pct_change()` tương đương `mau / mau.shift(1) - 1`. Nhân 100 nếu cần số phần trăm; định dạng `:.1%` cũng tự nhân 100 khi hiển thị. Không nhân thêm 100 trước định dạng phần trăm. Nếu kỳ gốc bằng 0, phép chia cho ra `inf` nên tỷ lệ không xác định.

In [ ]:
print((mau.pct_change() * 100).round(1))

### So với tháng liền trước

Tháng 4/2026 có 21.656 đánh giá; tháng 5 có 22.296. Kỳ gốc là tháng 4/2026. Chuỗi đã có một hàng mỗi tháng và đã bỏ tháng 6 chưa trọn. Dùng nhãn đủ ngày `"2026-05-31"` để lấy một giá trị; `.loc["2026-05"]` trả về Series một phần tử nên không định dạng được bằng `:.1%`.

In [ ]:
doi_thang = thang_day_du.pct_change()
print(f"Tháng 5/2026: {doi_thang.loc['2026-05-31']:.1%}")

### So với cùng tháng năm trước

Tháng 5/2025 có 14.509 đánh giá, tháng 5/2026 có 22.296; kỳ gốc là tháng 5/2025. Có thể viết cùng phép tính bằng `thang_day_du.pct_change(12)`. `12` là số hàng để lùi kỳ gốc; chuỗi phải giữ mọi tháng liên tiếp.

In [ ]:
doi_nam = thang_day_du / thang_day_du.shift(12) - 1
print(f"Tháng 5/2026: {doi_nam.loc['2026-05-31']:.1%}")

### Chọn kỳ gốc để so sánh

+3,0% so tháng 4/2026 và +53,7% so tháng 5/2025 đều đúng; hai kết quả dùng hai kỳ gốc khác nhau. Cần ghi rõ kỳ gốc. So cùng tháng giúp giảm ảnh hưởng của các khác biệt theo mùa, nhưng không tự giải thích nguyên nhân biến động.

### Tổng hợp theo tháng qua nhiều năm

Để xem số đánh giá thường phân bố thế nào theo tháng, gộp cùng tháng qua bốn năm gần nhất đủ 12 tháng (2022–2025), nên tháng nào cũng gộp từ bốn năm. Không dùng 2026 vì mới có dữ liệu đến 29/06: tháng 1–6 sẽ gộp từ 5 năm còn tháng 7–12 chỉ từ 4 năm. `tron` có ngày làm chỉ mục (`DatetimeIndex`) nên lấy tháng bằng `tron.index.month`; `.dt.month` chỉ dùng cho cột hoặc Series ngày giờ. Chỉ số của một tháng = tổng đánh giá của tháng đó qua bốn năm, chia cho trung bình của 12 tổng như vậy, rồi nhân 100; 100 là mức trung bình.

In [ ]:
tron = r.loc["2022":"2025"]
dem_thang = tron.groupby(tron.index.month).size()
chi_so = dem_thang / dem_thang.mean() * 100
print(chi_so.round().astype(int).tolist())

### Phân bố đánh giá theo tháng

Tổng đánh giá cả năm tăng từ 48.387 (2022) lên 211.432 (2025), trung bình khoảng 1,6 lần mỗi năm, nên các tháng cuối năm cao hơn một phần chỉ vì xu hướng. Chỉ số này chưa tách xu hướng và chưa điều chỉnh số ngày từng tháng. Ô vẽ bên dưới chỉ để quan sát; cách vẽ học ở Bài 12.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
chi_so.plot.bar(ax=ax, color="#1E93AB")
ax.axhline(100, color="0.4", linestyle="--")
ax.set_xlabel("Tháng")
ax.set_ylabel("Chỉ số (100 = trung bình)")
plt.show()

### Bài tập 3: So tổng số đánh giá

Dùng `thang_day_du` và `doi_nam` đã tạo. Muốn so **tổng số đánh giá 5 tháng đầu 2026** với cùng kỳ 2025, một bạn lấy trung bình của 5 tỷ lệ:

```python
thay_doi = doi_nam.loc["2026-01":"2026-05"].mean()
```

**Viết lại phép tính để đo mức thay đổi của tổng số đánh giá trong hai giai đoạn.**

In [ ]:
# TODO: viết lời giải ở đây.

## Làm việc với AI thì sao?

AI có thể gợi ý cú pháp, nhưng bạn phải nêu rõ trong yêu cầu: định dạng ngày, phạm vi dữ liệu,
kỳ gốc, cách gộp và cửa sổ cần tính. AI cũng có thể viết mã tần suất cũ như `"M"` mà pandas 3 báo lỗi.
Nếu bỏ `format`, pandas tự đoán định dạng từ giá trị đầu tiên và có thể vẫn chạy khi đoán nhầm.
Hãy kiểm bằng ngày dễ nhầm đặt ở hàng đầu tiên, ngày không tồn tại, lịch thiếu
một tháng và kỳ cuối chưa trọn.

Đối chiếu một tháng với dữ liệu gốc và tính lại kết quả cuối từ số đếm gốc trước khi dùng kết luận.

In [ ]:
assert len(rv_raw) == 690112
assert len(ngoai_moc) == 204
assert theo_thang.sum() == len(r) == 689908
assert theo_thang.loc["2026-03-31"] == len(r.loc["2026-03"])
assert thang_day_du.index[-1] == pd.Timestamp("2026-05-31")
assert dem_thang.sum() == len(tron)
print("Các phép đếm khớp với dữ liệu gốc.")

## Tự học · Múi giờ và cửa sổ theo khoảng thời gian

Nội dung dưới bổ sung §11.3–§11.4 và §11.7, không cần dùng để giải ba bài tập trên.

### Gắn múi giờ và đổi múi giờ

`tz_localize` gắn múi giờ cho giá trị chưa có múi giờ, theo quy ước của nguồn.
`tz_convert` đổi cách hiển thị của một thời điểm đã có múi giờ.

In [ ]:
gio = pd.to_datetime(pd.Series(["2026-06-29 22:30"]))
gio_utc = gio.dt.tz_localize("UTC")  # nguồn cho biết giờ là UTC
gio_vn = gio_utc.dt.tz_convert("Asia/Ho_Chi_Minh")
print(pd.DataFrame({"UTC": gio_utc, "VN": gio_vn}))

### Ba hàng khác với ba ngày

`rolling(3)` dùng ba hàng. `rolling("3D")` dùng khoảng ba ngày tính đến thời điểm
hiện tại, với biên trái không lấy và biên phải có lấy theo mặc định.
Chỉ mục phải là ngày giờ và đã sắp xếp. Cửa sổ theo thời gian mặc định cho phép
tính từ một giá trị, nên ví dụ viết rõ `min_periods=1` cho cả hai cách.

In [ ]:
lech_ngay = pd.Series([2, 1, 4], index=pd.to_datetime([
    "2026-06-01", "2026-06-03", "2026-06-07",
]))
print(pd.DataFrame({
    "tong_3_hang": lech_ngay.rolling(3, min_periods=1).sum(),
    "tong_3_ngay": lech_ngay.rolling("3D", min_periods=1).sum(),
}))

### Chọn khoảng giờ bằng biên phải mở

Nếu dữ liệu có cả giờ, điều kiện `<= "2026-06-29"` chỉ lấy đến 00:00 ngày 29/6.
Để lấy trọn ngày 29/6, dùng biên phải `< "2026-06-30"`, sau khi thống nhất múi giờ.

In [ ]:
nhat_ky = pd.Series(pd.to_datetime([
    "2026-06-29 00:00", "2026-06-29 18:30", "2026-06-30 00:00",
]))
trong_ngay = (nhat_ky >= "2026-06-29") & (nhat_ky < "2026-06-30")
print(nhat_ky[trong_ngay])

---

## Tóm tắt bài học

- Ngày giờ: xác định định dạng, kiểm lỗi chuyển đổi rồi tính khoảng thời gian.
- Tổng hợp: chọn lịch và phép tính; phân biệt ngày có 0 đánh giá với ngày chưa có dữ liệu.
- So sánh: giữ đúng kỳ gốc và phạm vi; `shift` dịch theo hàng, không theo lịch.
- Kỳ chưa trọn và tháng chưa thu thập dữ liệu cần được nhận diện trước khi tính.

Hoàn thành các bài tập rồi chạy **Restart session and run all**. Giữ phần giải thích
của bạn bên cạnh kết quả, đặc biệt khi code vẫn chạy nhưng trả lời sai yêu cầu.

**Đọc thêm:** [McKinney chương 11](https://wesmckinney.com/book/time-series),
[pandas: dữ liệu thời gian](https://pandas.pydata.org/docs/user_guide/timeseries.html),
[to_datetime](https://pandas.pydata.org/docs/reference/api/pandas.to_datetime.html),
[resample](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.resample.html),
[rolling](https://pandas.pydata.org/docs/reference/api/pandas.Series.rolling.html),
[shift](https://pandas.pydata.org/docs/reference/api/pandas.Series.shift.html),
[pct_change](https://pandas.pydata.org/docs/reference/api/pandas.Series.pct_change.html).

**Bài sau:** thi giữa kỳ; sau đó học làm sạch dữ liệu có cấu trúc ở Bài 10.